# 🤖 Model Training — SVM Cats vs Dogs Classifier

**PRODIGY_ML_03** | Prodigy InfoTech — ML Internship Task 03

This notebook trains a **Support Vector Machine (SVM)** classifier to distinguish between cat and dog images using:
- HOG (Histogram of Oriented Gradients) feature extraction
- StandardScaler normalization
- GridSearchCV for hyperparameter tuning

**Dataset:** [Kaggle Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats/data)

## 1. Import Libraries

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)
from skimage.feature import hog
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)

print('All libraries imported successfully!')

## 2. Configuration

In [ ]:
# Configuration
TARGET_SIZE = (64, 64)          # Image resize dimensions
MAX_SAMPLES_PER_CLASS = 2000    # Limit samples for reasonable training time
TEST_SIZE = 0.2                 # Train/test split ratio
RANDOM_STATE = 42               # Reproducibility

# Dataset path — update this to your dataset location
DATASET_PATH = os.path.join('..', '..', '..', 'artifacts', 'raw_data', 'train')

# Artifact paths
MODEL_PATH = os.path.join('..', '..', '..', 'artifacts', 'model.pkl')
SCALER_PATH = os.path.join('..', '..', '..', 'artifacts', 'preprocessor.pkl')

print(f'Target Size: {TARGET_SIZE}')
print(f'Max Samples/Class: {MAX_SAMPLES_PER_CLASS}')
print(f'Test Size: {TEST_SIZE}')
print(f'Dataset Path: {DATASET_PATH}')

## 3. Load Dataset

In [ ]:
# Load image paths
cats_path = os.path.join(DATASET_PATH, 'cats')
dogs_path = os.path.join(DATASET_PATH, 'dogs')

if os.path.exists(cats_path):
    cat_images = [os.path.join(cats_path, f) for f in os.listdir(cats_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:MAX_SAMPLES_PER_CLASS]
    dog_images = [os.path.join(dogs_path, f) for f in os.listdir(dogs_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:MAX_SAMPLES_PER_CLASS]
else:
    all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    cat_images = [os.path.join(DATASET_PATH, f) for f in all_files if f.lower().startswith('cat')][:MAX_SAMPLES_PER_CLASS]
    dog_images = [os.path.join(DATASET_PATH, f) for f in all_files if f.lower().startswith('dog')][:MAX_SAMPLES_PER_CLASS]

print(f'Cat images loaded: {len(cat_images)}')
print(f'Dog images loaded: {len(dog_images)}')
print(f'Total: {len(cat_images) + len(dog_images)}')

## 4. Feature Extraction (HOG)

In [ ]:
def extract_hog_from_image(image_path, target_size=(64, 64)):
    """Read, resize, convert to grayscale, and extract HOG features."""
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.resize(img, target_size)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False,
        feature_vector=True
    )
    return features


print('Extracting HOG features...')
start_time = time.time()

features = []
labels = []
skipped = 0

# Process cat images
for i, img_path in enumerate(cat_images):
    feat = extract_hog_from_image(img_path, TARGET_SIZE)
    if feat is not None:
        features.append(feat)
        labels.append(0)  # 0 = Cat
    else:
        skipped += 1
    if (i + 1) % 500 == 0:
        print(f'  Cats: {i+1}/{len(cat_images)} processed')

# Process dog images
for i, img_path in enumerate(dog_images):
    feat = extract_hog_from_image(img_path, TARGET_SIZE)
    if feat is not None:
        features.append(feat)
        labels.append(1)  # 1 = Dog
    else:
        skipped += 1
    if (i + 1) % 500 == 0:
        print(f'  Dogs: {i+1}/{len(dog_images)} processed')

X = np.array(features)
y = np.array(labels)

elapsed = time.time() - start_time
print(f'\nFeature extraction complete in {elapsed:.1f}s')
print(f'Feature matrix shape: {X.shape}')
print(f'Labels shape: {y.shape}')
print(f'Skipped images: {skipped}')

## 5. Train/Test Split & Scaling

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Feature dimensions: {X_train.shape[1]}')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nScaling applied — Mean: {X_train_scaled.mean():.4f}, Std: {X_train_scaled.std():.4f}')

## 6. SVM Training with GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01],
    'kernel': ['rbf', 'linear']
}

print('Starting GridSearchCV...')
print(f'Parameter grid: {param_grid}')
print(f'Total combinations: {len(param_grid["C"]) * len(param_grid["gamma"]) * len(param_grid["kernel"])}')
print(f'With 3-fold CV: {len(param_grid["C"]) * len(param_grid["gamma"]) * len(param_grid["kernel"]) * 3} fits\n')

start_time = time.time()

svm = SVC(probability=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    svm,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(X_train_scaled, y_train)

elapsed = time.time() - start_time
print(f'\nGridSearchCV completed in {elapsed:.1f}s')
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_:.4f}')

## 7. Model Evaluation

In [ ]:
# Get best model
best_model = grid_search.best_estimator_

# Predict on test set
y_pred = best_model.predict(X_test_scaled)
y_prob = best_model.predict_proba(X_test_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Cat', 'Dog'])
cm = confusion_matrix(y_test, y_pred)

print(f'Test Accuracy: {accuracy * 100:.2f}%')
print(f'\nClassification Report:')
print('=' * 55)
print(report)

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'],
            annot_kws={'size': 16, 'fontweight': 'bold'},
            linewidths=2, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold', color='#a78bfa')
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='#8b5cf6', lw=2.5, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'w--', alpha=0.3, lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#8b5cf6')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold', color='#a78bfa')
axes[1].legend(loc='lower right', fontsize=11)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob[:, 1])

axes[2].plot(recall, precision, color='#3b82f6', lw=2.5, label='Precision-Recall')
axes[2].fill_between(recall, precision, alpha=0.15, color='#3b82f6')
axes[2].set_xlabel('Recall', fontsize=12)
axes[2].set_ylabel('Precision', fontsize=12)
axes[2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold', color='#a78bfa')
axes[2].legend(loc='lower left', fontsize=11)

plt.tight_layout()
plt.show()

## 8. GridSearchCV Results Analysis

In [ ]:
# Analyze all GridSearchCV results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results_sorted = cv_results.sort_values('rank_test_score')

# Display top 10 parameter combinations
display_cols = ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
print('Top 10 Parameter Combinations:')
print('=' * 80)
print(cv_results_sorted[display_cols].head(10).to_string(index=False))

# Visualize CV scores by kernel type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for kernel in ['rbf', 'linear']:
    mask = cv_results['param_kernel'] == kernel
    color = '#8b5cf6' if kernel == 'rbf' else '#3b82f6'
    axes[0].scatter(
        cv_results[mask]['param_C'],
        cv_results[mask]['mean_test_score'],
        s=100, color=color, alpha=0.7, label=kernel.upper(),
        edgecolors='white', linewidth=1
    )

axes[0].set_xlabel('C (Regularization)', fontsize=12)
axes[0].set_ylabel('Mean CV Score', fontsize=12)
axes[0].set_title('CV Score vs C by Kernel', fontsize=14, fontweight='bold', color='#a78bfa')
axes[0].set_xscale('log')
axes[0].legend(fontsize=11)

# Bar chart of top results
top_5 = cv_results_sorted.head(5)
param_labels = [str(p) for p in top_5['params']]
short_labels = [f"#{i+1}" for i in range(5)]

bars = axes[1].barh(short_labels, top_5['mean_test_score'], color='#8b5cf6',
                    edgecolor='white', linewidth=1, height=0.5)
axes[1].set_xlabel('Mean CV Score', fontsize=12)
axes[1].set_title('Top 5 Configurations', fontsize=14, fontweight='bold', color='#a78bfa')

for bar, score in zip(bars, top_5['mean_test_score']):
    axes[1].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontsize=11, color='white')

plt.tight_layout()
plt.show()

## 9. Sample Predictions

In [ ]:
# Visualize predictions on test samples
# We need to get the original image paths for the test set
all_image_paths = cat_images + dog_images
_, test_paths, _, _ = train_test_split(
    all_image_paths, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Show random predictions
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Sample Predictions on Test Set', fontsize=18, fontweight='bold', color='#a78bfa', y=1.02)

indices = np.random.choice(len(y_test), 10, replace=False)
class_names = {0: 'Cat 🐱', 1: 'Dog 🐶'}

for i, idx in enumerate(indices):
    row, col = divmod(i, 5)
    
    img = cv2.imread(test_paths[idx])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    actual = class_names[y_test[idx]]
    predicted = class_names[y_pred[idx]]
    confidence = max(y_prob[idx]) * 100
    correct = y_test[idx] == y_pred[idx]
    
    axes[row, col].imshow(img_rgb)
    
    title_color = '#34d399' if correct else '#f87171'
    mark = '✅' if correct else '❌'
    axes[row, col].set_title(
        f'{mark} Pred: {predicted}\nConf: {confidence:.1f}%',
        fontsize=10, color=title_color, fontweight='bold'
    )
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 10. Save Model Artifacts

In [ ]:
# Save the trained model
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_model, f)
print(f'Model saved to: {MODEL_PATH}')

# Save the scaler
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Scaler saved to: {SCALER_PATH}')

# Save metrics
metrics = {
    'accuracy': accuracy,
    'best_params': grid_search.best_params_,
    'best_cv_score': grid_search.best_score_,
    'classification_report': classification_report(y_test, y_pred, target_names=['Cat', 'Dog'], output_dict=True),
    'classification_report_text': report,
    'confusion_matrix': cm,
    'y_pred': y_pred,
    'y_test': y_test
}

metrics_path = os.path.join('..', '..', '..', 'artifacts', 'metrics.pkl')
with open(metrics_path, 'wb') as f:
    pickle.dump(metrics, f)
print(f'Metrics saved to: {metrics_path}')

print(f'\n{"=" * 50}')
print(f'✅ All artifacts saved successfully!')
print(f'  - Model: {MODEL_PATH}')
print(f'  - Scaler: {SCALER_PATH}')
print(f'  - Metrics: {metrics_path}')
print(f'\n  Final Accuracy: {accuracy * 100:.2f}%')
print(f'  Best Params: {grid_search.best_params_}')

## 11. Summary

### Results:
| Metric | Value |
|--------|-------|
| **Test Accuracy** | See above |
| **Best CV Score** | See above |
| **Best Kernel** | See above |
| **Best C** | See above |
| **Best Gamma** | See above |

### Key Takeaways:
1. **HOG + SVM** is effective for binary image classification tasks
2. **GridSearchCV** found optimal hyperparameters automatically
3. The model achieves **good accuracy** despite the simplicity of SVM on image data
4. **RBF kernel** typically outperforms linear for non-linear decision boundaries in HOG space
5. **StandardScaler** is essential — SVM is sensitive to feature magnitudes

### Next Steps:
- Try with more samples (if compute allows)
- Experiment with different HOG parameters
- Test with other feature extractors (LBP, SIFT)
- Deploy via the **Streamlit app** (`streamlit run app.py`)